# Phase 4 — function-mode KD (depth12/experts80)

End-to-end view of the **bounded-function** KD fix: cache stats, training val
curve, and HumanEval-TS n=40 on the distilled student. Compare to pruned-only
**0.475** and raw-file KD **0.0** (full-param and LoRA) in `plan.md`.

Artifacts live under `docs/debug/phase4_depth12_experts80_functions_*` (committed
from the GPU runs). Eval: `tools/run_humaneval_ts_sample.sh`, same settings as
every other arm (temp 0.2, top-p 0.95, batch 8, limit 40).

In [1]:
import json
from pathlib import Path

DEBUG = Path("../docs/debug")
PREFIX = "phase4_depth12_experts80_functions"

summary = json.loads((DEBUG / f"{PREFIX}_run_summary.json").read_text())
summary

{'cache_wall_clock_sec': 848, 'cache_wall_clock_human': '14m 8s (02:16:46Z–02:30:54Z)', 'token_mean_cache_log': 127, 'token_mean_decoded_hub': 126.76, 'token_median_decoded_hub': 77.0, 'char_mean_decoded_hub': 499.99, 'char_median_decoded_hub': 310.0, 'kd_wall_clock_sec': 16061, 'kd_wall_clock_human': '4h 27m 21s', 'val_first_step': 9, 'val_first': 4.1479528292020165, 'val_final_step': 2999, 'val_final': 0.40938449968894325, 'val_best_step': 2999, 'val_best': 0.40938449968894325}

In [ ]:
import matplotlib.pyplot as plt

vals = []
metrics_path = DEBUG / f"{PREFIX}_kd_metrics.jsonl"
for line in metrics_path.read_text().splitlines():
    row = json.loads(line)
    if "val_loss" in row:
        vals.append((row["step"], row["val_loss"]))

steps, losses = zip(*vals, strict=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, losses, linewidth=1)
ax.set_xlabel("step")
ax.set_ylabel("val KD loss")
ax.set_title("depth12/experts80 function-mode KD (warmup450, lr1e-4, 3k steps)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [3]:
MODEL = "sartajbhuvaji/bonsai-distilled-depth12-experts80-functions-vs-baseline-3k-warmup450-lr1e4"
he = json.loads((DEBUG / f"{PREFIX}_humaneval_ts_n40.json").read_text())
pass_at_1 = he["multiple-ts"]["pass@1"]

comparison = {
    "pruned_only": 0.475,
    "raw_file_full_param_kd": 0.0,
    "raw_file_lora_kd": 0.0,
    "function_mode_kd": pass_at_1,
}
comparison

{'pruned_only': 0.475, 'raw_file_full_param_kd': 0.0, 'raw_file_lora_kd': 0.0, 'function_mode_kd': 0.0}

In [ ]:
gens = json.loads((DEBUG / f"{PREFIX}_humaneval_generations_multiple-ts.json").read_text())

for i in range(3):
    text = gens[i][0]
    print("=" * 72)
    print(f"problem {i} (len={len(text)})")
    print(text[:1200])
    if len(text) > 1200:
        print("... [truncated for display] ...")